In [4]:

import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models, transforms, datasets
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import KFold
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
import pandas as pd
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Executando no dispositivo:", device)


Executando no dispositivo: cuda


In [5]:

DATA_DIR = "../data/crops/c2_lesao"
IMG_SIZE = 70
BATCH_SIZE = 32
EPOCHS = 3
NUM_FOLDS = 3

transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

dataset = datasets.ImageFolder(DATA_DIR, transform=transform)
class_names = dataset.classes
print("Classes:", class_names)


Classes: ['com_lesao', 'sem_lesao']


In [ ]:

kf = KFold(n_splits=NUM_FOLDS, shuffle=True, random_state=42)

all_accuracies, all_precisions, all_recalls, all_f1s, all_specificities = [], [], [], [], []

for fold_idx, (train_idx, val_idx) in enumerate(kf.split(dataset)):
    print(f"\n===== FOLD {fold_idx+1} =====")

    train_subset = Subset(dataset, train_idx)
    val_subset = Subset(dataset, val_idx)

    # Augmentation no treino
    train_transform = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(10),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406],
                             [0.229, 0.224, 0.225])
    ])
    dataset_aug = datasets.ImageFolder(DATA_DIR, transform=train_transform)
    train_subset.dataset = dataset_aug

    print("Tamanho treino:", len(train_subset))
    print("Tamanho validação:", len(val_subset))

    train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_subset, batch_size=BATCH_SIZE, shuffle=False)

    model = models.efficientnet_b0(weights="IMAGENET1K_V1")
    model.classifier[1] = nn.Linear(model.classifier[1].in_features, 1)
    model = model.to(device)

    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.0003)

    model.train()
    for epoch in range(EPOCHS):
        loop = tqdm(train_loader, desc=f"Epoch [{epoch+1}/{EPOCHS} - Fold {fold_idx+1}]", leave=False)
        for inputs, labels in loop:
            inputs, labels = inputs.to(device), labels.float().unsqueeze(1).to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            loop.set_postfix(loss=loss.item())

    # Avaliação
    model.eval()
    
    y_true, y_pred = [], []
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs = inputs.to(device)
            outputs = model(inputs)
            preds = torch.sigmoid(outputs).cpu().numpy() > 0.5
            y_pred.extend(preds.astype(int).flatten())
            y_true.extend(labels.numpy())

    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred)
    rec = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    spec = tn / (tn + fp)

    print(f"Acurácia      : {acc:.4f}")
    print(f"Precisão      : {prec:.4f}")
    print(f"Revocação     : {rec:.4f}")
    print(f"F1-Score      : {f1:.4f}")
    print(f"Especificidade: {spec:.4f}")

    all_accuracies.append(acc)
    all_precisions.append(prec)
    all_recalls.append(rec)
    all_f1s.append(f1)
    all_specificities.append(spec)

    # Salvar matriz de confusão
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(4, 3))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
    plt.title(f'Matriz de Confusão - Fold {fold_idx+1}')
    plt.xlabel('Predito')
    plt.ylabel('Real')
    plt.tight_layout()
    plt.savefig(f'../results/c2/matriz_confusao_fold_{fold_idx+1}.png')
    plt.close()



===== FOLD 1 =====
Tamanho treino: 7689
Tamanho validação: 3845


Acurácia      : 0.9163
Precisão      : 0.9103
Revocação     : 0.9487
F1-Score      : 0.9291
Especificidade: 0.8717

===== FOLD 2 =====
Tamanho treino: 7689
Tamanho validação: 3845


Acurácia      : 0.9111
Precisão      : 0.9352
Revocação     : 0.9121
F1-Score      : 0.9235
Especificidade: 0.9096

===== FOLD 3 =====
Tamanho treino: 7690
Tamanho validação: 3844


Acurácia      : 0.9076
Precisão      : 0.9227
Revocação     : 0.9223
F1-Score      : 0.9225
Especificidade: 0.8860


In [ ]:
torch.save(model.state_dict(), '../models/classificador2.pth')
# Resultados finais
mean_acc = np.mean(all_accuracies)
mean_prec = np.mean(all_precisions)
mean_rec = np.mean(all_recalls)
mean_f1 = np.mean(all_f1s)
mean_spec = np.mean(all_specificities)

print("\n==== MÉDIAS FINAIS ====")
print(f"Acurácia média      : {mean_acc:.4f}")
print(f"Precisão média      : {mean_prec:.4f}")
print(f"Revocação média     : {mean_rec:.4f}")
print(f"F1-Score médio      : {mean_f1:.4f}")
print(f"Especificidade média: {mean_spec:.4f}")

df_resultados = pd.DataFrame({
    "Fold": list(range(1, NUM_FOLDS + 1)),
    "Acuracia": all_accuracies,
    "Precisao": all_precisions,
    "Revocacao": all_recalls,
    "F1": all_f1s,
    "Especificidade": all_specificities
})
df_resultados.loc["Media"] = df_resultados.mean(numeric_only=True)
df_resultados.to_csv("../results/c2/metricas_folds.csv", index=False)
print("Resultados salvos em resultados_classificador2.csv")



==== MÉDIAS FINAIS ====
Acurácia média      : 0.9117
Precisão média      : 0.9227
Revocação média     : 0.9277
F1-Score médio      : 0.9250
Especificidade média: 0.8891
Resultados salvos em resultados_classificador2.csv
